# Notebook 99 — Reproducibility Guard

Slim audit verifying every published headline number across Papers 1 and 2
against cached artifacts in `data/published/` and `results/`.
No re-execution; this is a check, not a fit. Runs in seconds.

A **FAIL** is information, not a build break — it signals drift between a
documented headline and its source artifact and should be investigated.
All failures are printed at the end; the notebook does not raise.

## §0 — Setup

In [ ]:
from __future__ import annotations

import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent.resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT / "notebooks"))

import matplotlib
matplotlib.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "figure.dpi":        120,
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
})

checks: list[dict] = []

def assert_close(
    actual: float,
    expected: float,
    label: str,
    source: str,
    paper: str = "",
    tol: float = 1e-3,
) -> None:
    delta = abs(actual - expected)
    status = "PASS" if delta <= tol else "FAIL"
    checks.append({
        "Paper":    paper,
        "Check":    label,
        "Expected": round(expected, 6),
        "Actual":   round(actual, 6),
        "Delta":    round(delta, 6),
        "Status":   status,
        "Source":   source,
    })

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

## §1 — Paper 1: Full-sample 2003–2026

Source: `data/published/master_table_62strategies.csv` (canonical Sharpes)
and `data/published/strategy_returns_base.parquet` /
`strategy_returns_vmp.parquet` (return series for computed checks).

Strategy names in the published CSV use `ledoit_wolf` rather than `LW`;
e.g. `VMP(MDP(ledoit_wolf))` = `VMP(MDP(LW))` in Paper 1 text.

In [ ]:
# ── Load Paper 1 published artifacts ───────────────────────────────────────
master = pd.read_csv(ROOT / "data/published/master_table_62strategies.csv")
base   = pd.read_parquet(ROOT / "data/published/strategy_returns_base.parquet")
vmp    = pd.read_parquet(ROOT / "data/published/strategy_returns_vmp.parquet")

SQRT252 = np.sqrt(252)

def ann_sharpe(s: pd.Series) -> float:
    s = s.dropna()
    return (s.mean() / s.std() * SQRT252) if s.std() > 0 else np.nan

def master_sharpe(strategy: str) -> float:
    row = master[master.Strategy == strategy]
    if row.empty:
        raise KeyError(strategy)
    return float(row.Sharpe.iloc[0])

SRC_MASTER = "data/published/master_table_62strategies.csv"
SRC_VMP_RET = "data/published/strategy_returns_vmp.parquet"
SRC_BASE_RET = "data/published/strategy_returns_base.parquet"

print(f"master_table: {len(master)} rows")
print(f"base returns : {base.shape} ({base.index[0].date()} → {base.index[-1].date()})")
print(f"vmp  returns : {vmp.shape}")

In [ ]:
# ── Check 1: VMP(MDP(LW)) — full-sample canonical top non-degenerate ───────
# Documented: 1.372 (Paper 1 text / notebook 00 headline findings)
# Source: master_table row 'VMP(MDP(ledoit_wolf))'
_actual = master_sharpe("VMP(MDP(ledoit_wolf))")
assert_close(_actual, 1.372, "VMP(MDP(LW)) canonical Sharpe", SRC_MASTER, paper="Paper 1")
print(f"VMP(MDP(LW))  master_table = {_actual:.6f}  (documented 1.372)")

# ── Check 2: VMP(GMV(sample)) — SHY-concentration artifact ────────────────
_actual2 = master_sharpe("VMP(GMV(sample))")
assert_close(_actual2, 1.345, "VMP(GMV(sample)) SHY-artifact Sharpe", SRC_MASTER, paper="Paper 1")
print(f"VMP(GMV(sample))  master_table = {_actual2:.6f}  (documented 1.345)")

# ── Check 3: VMP(SWITCH(v2a)) — best-in-study OOS regime rule ─────────────
# This strategy is not in the 62-strategy master table.
# We compute it from the cached switch_v2a return series + VMP overlay.
# Documented: 1.660 (notebook 00 headline findings)
from _shared import build_switch_v2a, load_base_returns
from aiam.evaluation.vmp_assembly import assemble_vmp_returns

_base_ret = load_base_returns()
_sw_oos   = pd.read_parquet(
    ROOT / "data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet"
)
_sw       = build_switch_v2a(_base_ret, _sw_oos)          # SWITCH(v2a)
_vmp_sw   = assemble_vmp_returns(_sw)                      # VMP overlay
_actual3  = ann_sharpe(_vmp_sw)
assert_close(
    _actual3, 1.660,
    "VMP(SWITCH(v2a)) best-in-study Sharpe",
    "data/cache/portfolio_returns/switch_v2a_oos_29assets.parquet + VMP overlay",
    paper="Paper 1",
)
print(f"VMP(SWITCH(v2a)) computed = {_actual3:.6f}  (documented 1.660)")

# ── Check 4: 24/24 VMP sign test (core 6 families) ────────────────────────
canon = pd.read_csv(ROOT / "data/cache/appendix_a_canonical.csv")
_core_fams = [
    "Classical MV", "Diversification", "Regime Switch",
    "TS Momentum", "Black-Litterman", "Factor",
]
_core_base = canon[canon.family.isin(_core_fams) & ~canon.is_vmp]
_wins = 0
for _, row in _core_base.iterrows():
    vmp_row = canon[canon.strategy == f"VMP({row.strategy})"]
    if not vmp_row.empty and vmp_row.iloc[0].sharpe >= row.sharpe:
        _wins += 1

assert_close(
    float(_wins), 24.0,
    "VMP sign test wins (core 24 strategies)",
    "data/cache/appendix_a_canonical.csv",
    paper="Paper 1",
    tol=0.5,  # integer count; pass if wins == 24
)
print(f"VMP sign test: {_wins}/24  (documented 24/24)")

## §2 — Paper 2 On-Axis: 29-asset OOS 2023-01-01 → 2026-04-30

Source: `data/published/full_comparison_with_rl.csv` (40-strategy dataset)
and `results/llm/llm-anthropic_diagnostics.json` for the LLM-BL row.

**All strategies below share the same evaluation window and harness** —
directly comparable.

In [ ]:
# ── Load Paper 2 on-axis published artifact ─────────────────────────────────
rl_csv = ROOT / "data/published/full_comparison_with_rl.csv"
p2 = pd.read_csv(rl_csv, index_col=0)
SRC_P2 = "data/published/full_comparison_with_rl.csv"

# LLM-BL from diagnostics JSON
with open(ROOT / "results/llm/llm-anthropic_diagnostics.json") as f:
    _llm_diag = json.load(f)
SRC_LLM = "results/llm/llm-anthropic_diagnostics.json"

print(f"P2 dataset   : {len(p2)} strategies")
print(f"LLM-Anthropic: Sharpe={_llm_diag['sharpe_ratio']:.4f}, refits={_llm_diag['n_refits']}")

# OOS VMP(MDP(LW)) from the same CSV — different from the full-sample 1.372
_vmp_mdp_oos = float(p2.loc["VMP(MDP(LW))", "Sharpe"])
print(f"OOS VMP(MDP(LW)): {_vmp_mdp_oos:.4f}  (full-sample: 1.3718 — different windows)")

In [ ]:
# ── Paper 2 on-axis checks ──────────────────────────────────────────────────
PAPER2_ONAXIS = [
    # (strategy key in p2 index,   expected Sharpe, label)
    ("MSR(Ensemble_μ̂)",  2.579, "ML bar MSR(Ensemble_μ̂)"),
    ("VMP(MDP(LW))",      2.422, "Classical OOS VMP(MDP(LW))"),
    ("MSR(RF_μ̂)",         2.394, "ML MSR(RF_μ̂)"),
    ("MSR(MLP_μ̂)",        2.386, "DL best on-axis MSR(MLP_μ̂)"),
    ("SignalTilt(XGB)",   2.304, "ML SignalTilt(XGB)"),
    ("RL(PPO,lam=0.02)",  2.027, "RL PPO best"),
    ("RL(REINFORCE,lam=0.00)", 2.026, "RL REINFORCE best"),
]

for key, exp, label in PAPER2_ONAXIS:
    act = float(p2.loc[key, "Sharpe"])
    assert_close(act, exp, label, SRC_P2, paper="Paper 2 (on-axis)")
    print(f"  {label:<40s}  actual={act:.4f}  expected={exp:.3f}")

# LLM-BL from JSON diagnostics (not in full_comparison_with_rl.csv)
_llm_actual = _llm_diag["sharpe_ratio"]
assert_close(_llm_actual, 2.284, "LLM-BL BL-LLM(Opus/Anthropic)", SRC_LLM, paper="Paper 2 (on-axis)")
print(f"  {'LLM-BL BL-LLM(Opus/Anthropic)':<40s}  actual={_llm_actual:.4f}  expected=2.284")

## §3 — Paper 2 Off-Axis

**These results are NOT comparable to §2** — different universes, windows,
or problem setups. Reported as confirmatory evidence only.

| Notebook | Setup | Reference |
|---|---|---|
| 05 | DL direct-weight, 2-asset SPY+IEF, OOS 2024-07→2026-04 | results/notebook_05/comparison.csv |
| 05b | Cross-asset deep momentum, 29-asset, 2015–2026 | results/notebook_05b/comparison.csv |
| 06c | Single-asset DQN (SPY), 2-action, OOS 2023–2026 | results/notebook_06c/summary.csv |

In [ ]:
# ── Notebook 05 — 2-asset DL direct-weight ─────────────────────────────────
df_05 = pd.read_csv(ROOT / "results/notebook_05/comparison.csv")
SRC_05 = "results/notebook_05/comparison.csv"

# Best DL: LSTM_dailyrp_crra_shrinkage
_05_dl  = df_05[df_05.family.str.startswith("DL")].sort_values("sharpe", ascending=False).iloc[0]
_05_rp  = df_05[df_05.strategy == "RiskParity-21d"].iloc[0]

assert_close(float(_05_dl.sharpe),  1.240, "05 best DL Sharpe (LSTM_dailyrp_crra_shrinkage)",
             SRC_05, paper="Paper 2 (off-axis)")
assert_close(float(_05_rp.sharpe),  1.247, "05 Risk Parity reference",
             SRC_05, paper="Paper 2 (off-axis)")

print(f"  05 best DL  : {_05_dl.strategy}  Sharpe={_05_dl.sharpe:.4f}  (expected ~1.240)")
print(f"  05 RP ref   : {_05_rp.strategy}  Sharpe={_05_rp.sharpe:.4f}  (expected ~1.247)")

# ── Notebook 05b — cross-asset deep momentum ───────────────────────────────
df_05b = pd.read_csv(ROOT / "results/notebook_05b/comparison.csv")
SRC_05B = "results/notebook_05b/comparison.csv"

_05b_cross  = df_05b[df_05b.strategy == "DeepMom(CrossAsset)"].iloc[0]
_05b_pooled = df_05b[df_05b.strategy == "DeepMom(Pooled)"].iloc[0]
_05b_rp     = df_05b[df_05b.strategy == "RiskParity-21d"].iloc[0]
_05b_tsmom  = df_05b[df_05b.strategy == "TSMOM-12m"].iloc[0]

assert_close(float(_05b_cross.sharpe),  0.852, "05b DeepMom(CrossAsset)",
             SRC_05B, paper="Paper 2 (off-axis)")
assert_close(float(_05b_pooled.sharpe), 0.849, "05b DeepMom(Pooled)",
             SRC_05B, paper="Paper 2 (off-axis)")
assert_close(float(_05b_rp.sharpe),     0.989, "05b Risk Parity reference",
             SRC_05B, paper="Paper 2 (off-axis)", tol=2e-3)
assert_close(float(_05b_tsmom.sharpe),  0.650, "05b TSMOM-12m reference",
             SRC_05B, paper="Paper 2 (off-axis)")

print(f"  05b DeepMom(CrossAsset) : {_05b_cross.sharpe:.4f}  (expected 0.852)")
print(f"  05b DeepMom(Pooled)     : {_05b_pooled.sharpe:.4f}  (expected 0.849)")
print(f"  05b Risk Parity         : {_05b_rp.sharpe:.4f}  (expected 0.989)")
print(f"  05b TSMOM-12m           : {_05b_tsmom.sharpe:.4f}  (expected 0.650)")

# ── Notebook 06c — single-asset DQN ────────────────────────────────────────
df_06c = pd.read_csv(ROOT / "results/notebook_06c/summary.csv")
SRC_06C = "results/notebook_06c/summary.csv"

_06c_dqn = df_06c[df_06c.strategy == "DQN_ensemble"].iloc[0]
_06c_bh  = df_06c[df_06c.strategy == "BuyAndHold_SPY"].iloc[0]

assert_close(float(_06c_dqn.sharpe), 1.20, "06c DQN Sharpe",
             SRC_06C, paper="Paper 2 (off-axis)", tol=5e-3)
assert_close(float(_06c_bh.sharpe),  1.33, "06c Buy-and-Hold reference",
             SRC_06C, paper="Paper 2 (off-axis)", tol=5e-3)

print(f"  06c DQN ensemble : {_06c_dqn.sharpe:.4f}  (expected ~1.20)")
print(f"  06c Buy-and-Hold : {_06c_bh.sharpe:.4f}  (expected ~1.33)")

## §4 — Summary

In [ ]:
df_checks = pd.DataFrame(checks)
display(df_checks.style.map(
    lambda v: "background-color: #ffcccc" if v == "FAIL" else
              "background-color: #ccffcc" if v == "PASS" else "",
    subset=["Status"],
))

n_pass = (df_checks.Status == "PASS").sum()
n_fail = (df_checks.Status == "FAIL").sum()
n_total = len(df_checks)

print()
if n_fail == 0:
    print(f"ALL {n_total} CHECKS PASS ✓")
else:
    print(f"{n_pass}/{n_total} PASS — {n_fail} FAIL(s) below:")
    fails = df_checks[df_checks.Status == "FAIL"]
    for _, row in fails.iterrows():
        print(
            f"  FAIL  [{row.Paper}]  {row.Check}\n"
            f"        expected={row.Expected:.6f}  actual={row.Actual:.6f}  "
            f"delta={row.Delta:.6f}  source={row.Source}"
        )